# Hands-On 1: Environment and dataset exploration

Record a prediction before running each experiment.

## Environment setup

Create the course environment once, then select its kernel in Jupyter:

```bash
conda env create -f environment.yml
conda activate ml-course
python -m pip install -e .
python -m ipykernel install --user --name ml-course --display-name "ML Course"
jupyter lab
```

Run these commands from the extracted course folder. A lightweight alternative is a virtual environment with `python -m pip install -r requirements.txt` followed by `python -m pip install -e .`.

In Colab, make the extracted course folder available in the runtime or mounted Drive. Set `MLCOURSE_ROOT` to that folder if it is outside the current location. Run the first two collapsed cells; the shared setup installs only missing packages there.

The experiments below separate the environment check from dataset exploration.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from mlcourse.labs import histogram_grid
import importlib.metadata

## 1. Check the environment

Run the environment check. Confirm which Python executable the notebook kernel uses.

**Prediction:** Why might two notebooks on the same computer see different installed packages?

*Your response.*

In [ ]:
print(f'Python executable: {sys.executable}')
for package in ['numpy', 'pandas', 'matplotlib', 'scikit-learn']:
    print(f'{package}: {importlib.metadata.version(package)}')

**Observation:** Confirm that the executable belongs to your chosen environment.

*Your response.*

**Explanation:** What is the relationship between a kernel and an environment?

*Your response.*

## 2. Inspect a dataframe

Read the reservations dataset. Inspect `head()`, `dtypes` and `describe()`.

**Prediction:** Will every numeric-looking column represent a continuous quantity?

*Your response.*

In [ ]:
data = load_course_data('hotel')
display(data.head())
print(f'Rows: {data.shape[0]}; columns: {data.shape[1]}')
display(data.dtypes.to_frame('dtype'))
display(data.describe())

**Observation:** Identify one count, one measurement and one category.

*Your response.*

**Explanation:** Why distinguish storage type from the meaning of a feature?

*Your response.*

## 3. Select and filter observations

Inspect selected columns, compute a mean and filter rows using two conditions.

**Prediction:** How should combining conditions with `&` change the selected rows?

*Your response.*

In [ ]:
display(data[['room_type_reserved', 'required_car_parking_space']].head(10))
display(data.room_type_reserved.value_counts().rename('count'))
print(f'Mean lead time: {data.lead_time.mean():.3f}')
selected = data.loc[(data.no_of_previous_cancellations > 0) & (data.no_of_children > 0),
                    ['room_type_reserved', 'no_of_children', 'no_of_previous_cancellations']]
print(f'Selected rows: {len(selected)}')
display(selected.head())
display(data.nlargest(5, 'avg_price_per_room')[['avg_price_per_room', 'booking_status']])

**Observation:** Verify that the shown rows satisfy both conditions.

*Your response.*

**Explanation:** Why can a summary of a filtered subset differ from the whole dataset?

*Your response.*

## 4. Group and cross-tabulate

Count bookings by status and segment. Summarise numeric features within each status.

**Prediction:** Could a frequent class dominate an overall average?

*Your response.*

In [ ]:
display(pd.crosstab(data.market_segment_type, data.booking_status))
display(data.groupby('booking_status')[['lead_time', 'avg_price_per_room']].mean())
fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
pd.crosstab(data.market_segment_type, data.booking_status).plot.bar(ax=ax, rot=20)
ax.set(xlabel='Segment', ylabel='Count', title='Class counts within each segment')
plt.show()

**Observation:** Compare class counts and group means.

*Your response.*

**Explanation:** What information does grouping preserve that one overall mean hides?

*Your response.*

## 5. Inspect feature distributions

Plot histograms for all numeric columns.

**Prediction:** Which columns might have many small values and a few large values?

*Your response.*

In [ ]:
histogram_grid(data)

**Observation:** Identify a skewed distribution and a discrete count distribution.

*Your response.*

**Explanation:** How could scale and skew affect a distance-based model?

*Your response.*

## 6. Compare class proportions and spread

Plot booking-status counts and lead-time distributions by status.

**Prediction:** Can different group means coexist with overlapping distributions?

*Your response.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
data.booking_status.value_counts().plot.bar(ax=axes[0], rot=15, color='#0072B2')
axes[0].set(xlabel='Status', ylabel='Count', title='Target distribution')
data.boxplot(column='lead_time', by='booking_status', ax=axes[1], grid=False)
axes[1].set(xlabel='Status', ylabel='Lead time', title='Lead time by target class')
fig.suptitle('')
plt.show()

**Observation:** Compare medians, spread and overlap.

*Your response.*

**Explanation:** Why does a difference between averages not imply perfect classification?

*Your response.*